# NexusTrade — 1-Year Comparison Analyzer
## Long-Term Trend & Structural Analysis

Analyseer 12 maanden prijsdata om dilution, trend structure, insider patterns en risk asymmetry te beoordelen.

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
from typing import List, Dict, Tuple, Optional

class OneYearAnalyzer:
    """
    Analyseert 12 maanden data voor long-term context:
    - Share dilution impact
    - Trend structure (higher highs/lows vs lower highs/lows)
    - Insider selling timeline
    - 52-week positioning
    - Risk asymmetry (upside vs downside)
    """
    
    def __init__(self, ticker: str):
        self.ticker = ticker
        self.annual_metrics = {}
        self.insider_timeline = []
        self.news_timeline = []
        
    def add_annual_metrics(self, 
                          week_52_high: float,
                          week_52_low: float,
                          current_price: float,
                          shares_outstanding_start: int,
                          shares_outstanding_end: int,
                          performance_ytd: float,
                          performance_year: float,
                          performance_half: float,
                          short_interest_pct: Optional[float] = None,
                          institutional_ownership_pct: Optional[float] = None):
        """
        Voeg jaarlijkse metrics toe
        
        Args:
            week_52_high: 52-week high
            week_52_low: 52-week low
            current_price: Huidige prijs
            shares_outstanding_start: Shares outstanding begin van jaar
            shares_outstanding_end: Shares outstanding einde van jaar
            performance_ytd: YTD performance %
            performance_year: 1-year performance %
            performance_half: Half-year performance %
            short_interest_pct: Short interest percentage (optioneel)
            institutional_ownership_pct: Institutional ownership % (optioneel)
        """
        distance_from_high = ((current_price - week_52_high) / week_52_high) * 100
        distance_from_low = ((current_price - week_52_low) / week_52_low) * 100
        
        self.annual_metrics = {
            '52w_high': week_52_high,
            '52w_low': week_52_low,
            'current_price': current_price,
            'range_position_pct': ((current_price - week_52_low) / (week_52_high - week_52_low)) * 100,
            'distance_from_high_pct': distance_from_high,
            'distance_from_low_pct': distance_from_low,
            'shares_start': shares_outstanding_start,
            'shares_end': shares_outstanding_end,
            'dilution_factor': shares_outstanding_end / shares_outstanding_start,
            'dilution_pct': ((shares_outstanding_end - shares_outstanding_start) / shares_outstanding_start) * 100,
            'perf_ytd': performance_ytd,
            'perf_year': performance_year,
            'perf_half': performance_half,
            'short_interest': short_interest_pct,
            'institutional_ownership': institutional_ownership_pct
        }
    
    def add_insider_event(self, date: str, price: float, 
                          shares: int, value: float,
                          entity: str, action: str = "SELL"):
        """
        Voeg insider trading event toe aan timeline
        
        Args:
            date: Datum (YYYY-MM-DD of Month YYYY)
            price: Transactie prijs
            shares: Aantal aandelen
            value: Totale waarde in dollars
            entity: Insider/entity naam
            action: BUY of SELL
        """
        self.insider_timeline.append({
            'date': date,
            'action': action,
            'price': price,
            'shares': shares,
            'value': value,
            'entity': entity
        })
    
    def add_news_catalyst(self, date: str, headline: str, 
                          price_impact_pct: float,
                          price_level: float):
        """
        Voeg news event toe
        
        Args:
            date: Datum
            headline: Nieuwskop
            price_impact_pct: Impact percentage
            price_level: Prijs niveau bij event
        """
        self.news_timeline.append({
            'date': date,
            'headline': headline,
            'impact_pct': price_impact_pct,
            'price_level': price_level
        })
    
    def calculate_dilution_impact(self) -> Dict[str, any]:
        """
        Bereken impact van share dilution op ownership
        
        Returns:
            Dilution analysis
        """
        if not self.annual_metrics:
            return {'error': 'No annual metrics added yet'}
        
        dilution_factor = self.annual_metrics['dilution_factor']
        dilution_pct = self.annual_metrics['dilution_pct']
        
        # Bereken ownership destruction
        # Als shares 10x gaan, is je ownership 10% van wat het was (= 90% destruction)
        ownership_retained_pct = (1 / dilution_factor) * 100
        ownership_destroyed_pct = 100 - ownership_retained_pct
        
        # Severity rating
        if dilution_factor < 1.2:
            severity = "MINIMAL (<20% dilution)"
        elif dilution_factor < 1.5:
            severity = "MODERATE (20-50% dilution)"
        elif dilution_factor < 2.0:
            severity = "SIGNIFICANT (50-100% dilution)"
        elif dilution_factor < 5.0:
            severity = "SEVERE (2-5x dilution)"
        else:
            severity = "CATASTROPHIC (>5x dilution)"
        
        return {
            'shares_start': self.annual_metrics['shares_start'],
            'shares_end': self.annual_metrics['shares_end'],
            'dilution_factor': round(dilution_factor, 2),
            'dilution_pct': round(dilution_pct, 1),
            'ownership_retained_pct': round(ownership_retained_pct, 1),
            'ownership_destroyed_pct': round(ownership_destroyed_pct, 1),
            'severity': severity,
            'interpretation': f"Each original share now represents {ownership_retained_pct:.1f}% of its original ownership"
        }
    
    def analyze_trend_structure(self, key_highs: List[Tuple[str, float]], 
                                 key_lows: List[Tuple[str, float]]) -> Dict[str, any]:
        """
        Analyseer trend structure: higher highs/lows vs lower highs/lows
        
        Args:
            key_highs: List van (datum, prijs) tuples voor belangrijke highs
            key_lows: List van (datum, prijs) tuples voor belangrijke lows
            
        Returns:
            Trend structure analysis
        """
        # Check high trend
        high_prices = [h[1] for h in key_highs]
        high_trend = "LOWER HIGHS (bearish)" if all(high_prices[i] >= high_prices[i+1] for i in range(len(high_prices)-1)) else \
                     "HIGHER HIGHS (bullish)" if all(high_prices[i] <= high_prices[i+1] for i in range(len(high_prices)-1)) else \
                     "MIXED/SIDEWAYS"
        
        # Check low trend
        low_prices = [l[1] for l in key_lows]
        low_trend = "LOWER LOWS (bearish)" if all(low_prices[i] >= low_prices[i+1] for i in range(len(low_prices)-1)) else \
                    "HIGHER LOWS (bullish)" if all(low_prices[i] <= low_prices[i+1] for i in range(len(low_prices)-1)) else \
                    "MIXED/SIDEWAYS"
        
        # Overall trend determination
        if "bearish" in high_trend and "bearish" in low_trend:
            overall_trend = "STRONG DOWNTREND"
        elif "bullish" in high_trend and "bullish" in low_trend:
            overall_trend = "STRONG UPTREND"
        elif "bearish" in high_trend:
            overall_trend = "WEAK DOWNTREND (failing rallies)"
        elif "bullish" in low_trend:
            overall_trend = "WEAK UPTREND (higher lows but limited upside)"
        else:
            overall_trend = "SIDEWAYS/CONSOLIDATION"
        
        return {
            'high_trend': high_trend,
            'low_trend': low_trend,
            'overall_trend': overall_trend,
            'key_highs': key_highs,
            'key_lows': key_lows,
            'highest_point': max(high_prices),
            'lowest_point': min(low_prices),
            'annual_range': max(high_prices) - min(low_prices),
            'range_contraction_pct': ((key_highs[-1][1] - key_lows[-1][1]) / (key_highs[0][1] - key_lows[0][1]) * 100) if len(key_highs) > 1 else 100
        }
    
    def insider_selling_analysis(self) -> Dict[str, any]:
        """
        Analyseer insider selling timeline en patronen
        
        Returns:
            Insider selling analysis
        """
        if not self.insider_timeline:
            return {'status': 'No insider events recorded'}
        
        df = pd.DataFrame(self.insider_timeline)
        
        # Filter sells only
        sells = df[df['action'] == 'SELL']
        
        if len(sells) == 0:
            return {'status': 'No insider sells found'}
        
        total_shares_sold = sells['shares'].sum()
        total_value_sold = sells['value'].sum()
        avg_sell_price = sells['price'].mean()
        
        # Check if selling accelerated over time
        sells_sorted = sells.sort_values('date')
        first_half_volume = sells_sorted.head(len(sells)//2)['shares'].sum()
        second_half_volume = sells_sorted.tail(len(sells)//2)['shares'].sum()
        
        acceleration = "ACCELERATING" if second_half_volume > first_half_volume * 1.5 else \
                       "STEADY" if second_half_volume >= first_half_volume * 0.75 else \
                       "DECLINING"
        
        return {
            'total_sell_events': len(sells),
            'total_shares_sold': total_shares_sold,
            'total_value_sold': f"${total_value_sold:,.0f}",
            'avg_sell_price': round(avg_sell_price, 4),
            'highest_sell_price': sells['price'].max(),
            'lowest_sell_price': sells['price'].min(),
            'selling_pattern': acceleration,
            'timeline': sells[['date', 'entity', 'shares', 'price', 'value']].to_dict('records')
        }
    
    def assess_52w_position(self) -> Dict[str, any]:
        """
        Beoordeel huidige positie in 52-week range
        
        Returns:
            52-week positioning analysis
        """
        if not self.annual_metrics:
            return {'error': 'No annual metrics added'}
        
        metrics = self.annual_metrics
        position_pct = metrics['range_position_pct']
        
        # Determine position quality
        if position_pct <= 20:
            position_quality = "EXCELLENT (bottom 20% of range)"
            bias = "Strong upside potential"
        elif position_pct <= 40:
            position_quality = "GOOD (lower 40% of range)"
            bias = "Favorable upside bias"
        elif position_pct <= 60:
            position_quality = "NEUTRAL (mid-range)"
            bias = "Balanced risk/reward"
        elif position_pct <= 80:
            position_quality = "POOR (upper 60-80%)"
            bias = "Limited upside, elevated risk"
        else:
            position_quality = "VERY POOR (top 20% of range)"
            bias = "High risk, minimal upside"
        
        return {
            '52w_high': metrics['52w_high'],
            '52w_low': metrics['52w_low'],
            'current_price': metrics['current_price'],
            'range_position_pct': round(position_pct, 1),
            'position_quality': position_quality,
            'bias': bias,
            'distance_from_high': f"{metrics['distance_from_high_pct']:.1f}%",
            'distance_from_low': f"+{metrics['distance_from_low_pct']:.1f}%",
            'perf_year': f"{metrics['perf_year']:.2f}%",
            'perf_half': f"{metrics['perf_half']:.2f}%",
            'perf_ytd': f"{metrics['perf_ytd']:.2f}%"
        }
    
    def calculate_risk_asymmetry(self, upside_target: float, 
                                  downside_support: float,
                                  worst_case_scenario: Optional[float] = None) -> Dict[str, any]:
        """
        Bereken risk/reward asymmetry
        
        Args:
            upside_target: Verwachte upside target
            downside_support: Nearest support level
            worst_case_scenario: Worst-case prijs (optioneel, default = 52w low)
            
        Returns:
            Risk asymmetry analysis
        """
        if not self.annual_metrics:
            return {'error': 'No annual metrics added'}
        
        current = self.annual_metrics['current_price']
        
        if worst_case_scenario is None:
            worst_case_scenario = self.annual_metrics['52w_low']
        
        # Calculate potential moves
        upside_pct = ((upside_target - current) / current) * 100
        downside_to_support_pct = ((downside_support - current) / current) * 100
        downside_to_worst_pct = ((worst_case_scenario - current) / current) * 100
        
        # Risk/Reward ratios
        rr_to_support = abs(upside_pct / downside_to_support_pct) if downside_to_support_pct != 0 else None
        rr_to_worst = abs(upside_pct / downside_to_worst_pct) if downside_to_worst_pct != 0 else None
        
        # Assessment
        if rr_to_support and rr_to_support >= 2.5:
            assessment = "FAVORABLE (upside >2.5x downside to support)"
        elif rr_to_support and rr_to_support >= 1.5:
            assessment = "ACCEPTABLE (upside >1.5x downside to support)"
        elif rr_to_support and rr_to_support >= 1.0:
            assessment = "NEUTRAL (upside ~= downside)"
        else:
            assessment = "UNFAVORABLE (downside > upside)"
        
        return {
            'current_price': current,
            'upside_target': upside_target,
            'upside_pct': round(upside_pct, 2),
            'downside_support': downside_support,
            'downside_to_support_pct': round(downside_to_support_pct, 2),
            'worst_case': worst_case_scenario,
            'downside_to_worst_pct': round(downside_to_worst_pct, 2),
            'rr_ratio_to_support': round(rr_to_support, 2) if rr_to_support else None,
            'rr_ratio_to_worst': round(rr_to_worst, 2) if rr_to_worst else None,
            'assessment': assessment
        }
    
    def generate_full_report(self, key_highs: List[Tuple[str, float]], 
                            key_lows: List[Tuple[str, float]],
                            upside_target: float,
                            downside_support: float) -> Dict[str, any]:
        """
        Genereer volledig 1-year analysis report
        
        Returns:
            Complete analysis dictionary
        """
        return {
            'ticker': self.ticker,
            'annual_metrics': self.annual_metrics,
            'dilution_impact': self.calculate_dilution_impact(),
            'trend_structure': self.analyze_trend_structure(key_highs, key_lows),
            'insider_selling': self.insider_selling_analysis(),
            '52w_position': self.assess_52w_position(),
            'risk_asymmetry': self.calculate_risk_asymmetry(upside_target, downside_support)
        }

def compare_time_periods(ytd_perf: float, half_year_perf: float, 
                         year_perf: float) -> Dict[str, any]:
    """
    Vergelijk performance over verschillende tijdsperiodes
    
    Args:
        ytd_perf: Year-to-date performance %
        half_year_perf: Half-year performance %
        year_perf: Full year performance %
        
    Returns:
        Comparative analysis
    """
    # Determine trend direction
    if ytd_perf > 0 > half_year_perf:
        momentum = "RECOVERING (positive YTD but negative half-year)"
    elif ytd_perf > 0 and half_year_perf > 0:
        momentum = "BULLISH MOMENTUM (positive across timeframes)"
    elif ytd_perf < 0 and half_year_perf < 0:
        momentum = "BEARISH MOMENTUM (negative across timeframes)"
    elif ytd_perf < 0 < half_year_perf:
        momentum = "DETERIORATING (was positive, now negative)"
    else:
        momentum = "MIXED SIGNALS"
    
    # Calculate velocity (rate of change)
    recent_velocity = ytd_perf  # Recent trend
    longer_velocity = half_year_perf  # Longer trend
    
    velocity_change = recent_velocity - longer_velocity
    
    if velocity_change > 20:
        velocity_assessment = "ACCELERATING UPWARD"
    elif velocity_change > 5:
        velocity_assessment = "IMPROVING"
    elif velocity_change > -5:
        velocity_assessment = "STABLE"
    elif velocity_change > -20:
        velocity_assessment = "DECLINING"
    else:
        velocity_assessment = "ACCELERATING DOWNWARD"
    
    return {
        'ytd_perf': ytd_perf,
        'half_year_perf': half_year_perf,
        'year_perf': year_perf,
        'momentum': momentum,
        'velocity_change': round(velocity_change, 2),
        'velocity_assessment': velocity_assessment,
        'interpretation': f"Recent momentum is {velocity_assessment.lower()} compared to half-year trend"
    }

def calculate_long_term_probability(current_price: float, 
                                     target_price: float,
                                     key_highs: List[float],
                                     insider_avg_sell: float) -> Dict[str, any]:
    """
    Bereken waarschijnlijkheid van target bereiken op basis van historische context
    
    Args:
        current_price: Huidige prijs
        target_price: Doel prijs
        key_highs: Lijst van key highs uit het jaar
        insider_avg_sell: Gemiddelde insider sell prijs
        
    Returns:
        Probability assessment
    """
    # Check hoeveel key highs boven target zijn
    highs_above_target = sum(1 for h in key_highs if h > target_price)
    total_highs = len(key_highs)
    
    historical_hit_rate = (highs_above_target / total_highs) * 100 if total_highs > 0 else 0
    
    # Insider sell price als indicator
    if insider_avg_sell > target_price:
        insider_signal = "NEGATIVE (insiders sold above your target)"
    elif insider_avg_sell < target_price * 0.9:
        insider_signal = "POSITIVE (insiders sold well below target)"
    else:
        insider_signal = "NEUTRAL (insiders sold near target range)"
    
    # Overall probability
    distance_pct = ((target_price - current_price) / current_price) * 100
    
    if historical_hit_rate > 75 and "POSITIVE" in insider_signal:
        probability = "HIGH (75-90%)"
    elif historical_hit_rate > 50:
        probability = "MODERATE (50-75%)"
    elif historical_hit_rate > 25:
        probability = "LOW (25-50%)"
    else:
        probability = "VERY LOW (<25%)"
    
    return {
        'current_price': current_price,
        'target_price': target_price,
        'distance_to_target_pct': round(distance_pct, 2),
        'historical_hit_rate': f"{historical_hit_rate:.1f}%",
        'highs_above_target': highs_above_target,
        'total_highs_analyzed': total_highs,
        'insider_signal': insider_signal,
        'probability_assessment': probability
    }

## Example: DVLT 1-Year Analysis (Apr 2025 - Apr 2026)

Real-world voorbeeld met complete DVLT data over 12 maanden.

In [ ]:
# Initialize analyzer
analyzer = OneYearAnalyzer(ticker="DVLT")

# Add annual metrics
analyzer.add_annual_metrics(
    week_52_high=4.10,
    week_52_low=0.25,
    current_price=0.74,
    shares_outstanding_start=52_030_000,  # 2024
    shares_outstanding_end=573_440_000,   # 2025 — 10x dilution!
    performance_ytd=13.26,
    performance_year=0.76,
    performance_half=-70.81,
    short_interest_pct=22.17,
    institutional_ownership_pct=6.01
)

# Add insider selling timeline
analyzer.add_insider_event(
    date="Aug 2025",
    price=0.40,
    shares=5_000_000,
    value=2_000_000,
    entity="Scilex Early Sell"
)

analyzer.add_insider_event(
    date="Nov 2025",
    price=2.44,
    shares=3_000_000,
    value=7_320_000,
    entity="Scilex at Peak"
)

analyzer.add_insider_event(
    date="Jan 2026",
    price=1.37,
    shares=4_000_000,
    value=5_480_000,
    entity="Scilex"
)

analyzer.add_insider_event(
    date="Jan 2026",
    price=1.20,
    shares=3_500_000,
    value=4_200_000,
    entity="Scilex"
)

analyzer.add_insider_event(
    date="Jan 2026",
    price=0.88,
    shares=2_500_000,
    value=2_200_000,
    entity="Scilex Board Member"
)

analyzer.add_insider_event(
    date="Jan 2026",
    price=0.72,
    shares=5_000_000,
    value=3_600_000,
    entity="Scilex CFO"
)

analyzer.add_insider_event(
    date="Mar 2026",
    price=0.63,
    shares=10_000_000,
    value=6_300_000,
    entity="Scilex Holdings LLC"
)

print("═══════════════════════════════════════════════════════════════")
print("1-YEAR COMPARISON ANALYSIS — DVLT")
print("═══════════════════════════════════════════════════════════════\n")

In [ ]:
# Dilution Impact Analysis
print("SHARE DILUTION IMPACT:")
print("═══════════════════════════════════════════════════════════════")
dilution = analyzer.calculate_dilution_impact()

print(f"Shares Outstanding:")
print(f"  Start (2024): {dilution['shares_start']:,}")
print(f"  End (2025):   {dilution['shares_end']:,}")
print(f"\nDilution Factor: {dilution['dilution_factor']}x ({dilution['dilution_pct']}% increase)")
print(f"Ownership Retained: {dilution['ownership_retained_pct']}%")
print(f"Ownership Destroyed: {dilution['ownership_destroyed_pct']}%")
print(f"\nSEVERITY: {dilution['severity']}")
print(f"\n💡 {dilution['interpretation']}")
print()

In [ ]:
# Trend Structure Analysis
print("\nTREND STRUCTURE ANALYSIS:")
print("═══════════════════════════════════════════════════════════════")

# Define key highs and lows from the year
key_highs = [
    ("Apr 2025", 4.10),
    ("Nov 2025", 2.44),
    ("Jan 2026", 1.37),
    ("Apr 2026", 0.84)
]

key_lows = [
    ("Aug 2025", 0.25),
    ("Dec 2025", 0.64),
    ("Mar 2026", 0.60),
    ("Apr 2026", 0.64)
]

trend = analyzer.analyze_trend_structure(key_highs, key_lows)

print(f"High Trend: {trend['high_trend']}")
print(f"Low Trend: {trend['low_trend']}")
print(f"Overall Trend: {trend['overall_trend']}")
print(f"\nKEY HIGHS:")
for date, price in trend['key_highs']:
    print(f"  {date}: ${price}")
print(f"\nKEY LOWS:")
for date, price in trend['key_lows']:
    print(f"  {date}: ${price}")
print(f"\nHighest Point: ${trend['highest_point']}")
print(f"Lowest Point: ${trend['lowest_point']}")
print(f"Annual Range: ${trend['annual_range']:.2f}")
print(f"Range Contraction: {trend['range_contraction_pct']:.1f}%")
print()

In [ ]:
# Insider Selling Analysis
print("\nINSIDER SELLING TIMELINE:")
print("═══════════════════════════════════════════════════════════════")
insider = analyzer.insider_selling_analysis()

print(f"Total Sell Events: {insider['total_sell_events']}")
print(f"Total Shares Sold: {insider['total_shares_sold']:,}")
print(f"Total Value Sold: {insider['total_value_sold']}")
print(f"Avg Sell Price: ${insider['avg_sell_price']}")
print(f"Price Range: ${insider['lowest_sell_price']} - ${insider['highest_sell_price']}")
print(f"Selling Pattern: {insider['selling_pattern']}")

print(f"\nTIMELINE:")
timeline_df = pd.DataFrame(insider['timeline'])
timeline_df['value_display'] = timeline_df['value'].apply(lambda x: f"${x:,.0f}")
print(timeline_df[['date', 'entity', 'shares', 'price', 'value_display']].to_string(index=False))
print()

In [ ]:
# 52-Week Position Assessment
print("\n52-WEEK POSITION ASSESSMENT:")
print("═══════════════════════════════════════════════════════════════")
position = analyzer.assess_52w_position()

print(f"52-Week Range: ${position['52w_low']} - ${position['52w_high']}")
print(f"Current Price: ${position['current_price']}")
print(f"Range Position: {position['range_position_pct']}th percentile")
print(f"\nPosition Quality: {position['position_quality']}")
print(f"Bias: {position['bias']}")
print(f"\nDistance from High: {position['distance_from_high']}")
print(f"Distance from Low: {position['distance_from_low']}")
print(f"\nPERFORMANCE:")
print(f"  YTD:      {position['perf_ytd']}")
print(f"  Half Year: {position['perf_half']}")
print(f"  1-Year:    {position['perf_year']}")
print()

In [ ]:
# Risk Asymmetry Analysis
print("\nRISK ASYMMETRY ANALYSIS:")
print("═══════════════════════════════════════════════════════════════")

# User's breakeven and support levels
UPSIDE_TARGET = 0.88  # Realistic upside target
DOWNSIDE_SUPPORT = 0.60  # March low
WORST_CASE = 0.25  # 52-week low

risk_asymmetry = analyzer.calculate_risk_asymmetry(
    upside_target=UPSIDE_TARGET,
    downside_support=DOWNSIDE_SUPPORT,
    worst_case_scenario=WORST_CASE
)

print(f"Current Price: ${risk_asymmetry['current_price']}")
print(f"\nUPSIDE SCENARIO:")
print(f"  Target: ${risk_asymmetry['upside_target']}")
print(f"  Gain: +{risk_asymmetry['upside_pct']}%")
print(f"\nDOWNSIDE SCENARIO:")
print(f"  Support: ${risk_asymmetry['downside_support']}")
print(f"  Loss to Support: {risk_asymmetry['downside_to_support_pct']}%")
print(f"  Worst Case: ${risk_asymmetry['worst_case']}")
print(f"  Loss to Worst: {risk_asymmetry['downside_to_worst_pct']}%")
print(f"\nRISK/REWARD RATIOS:")
print(f"  To Support: {risk_asymmetry['rr_ratio_to_support']}:1")
print(f"  To Worst Case: {risk_asymmetry['rr_ratio_to_worst']}:1")
print(f"\nASSESSMENT: {risk_asymmetry['assessment']}")
print()

In [ ]:
# Time Period Comparison
print("\nTIME PERIOD COMPARISON:")
print("═══════════════════════════════════════════════════════════════")

period_comparison = compare_time_periods(
    ytd_perf=13.26,
    half_year_perf=-70.81,
    year_perf=0.76
)

print(f"YTD Performance: {period_comparison['ytd_perf']}%")
print(f"Half-Year Performance: {period_comparison['half_year_perf']}%")
print(f"Full Year Performance: {period_comparison['year_perf']}%")
print(f"\nMomentum: {period_comparison['momentum']}")
print(f"Velocity Change: {period_comparison['velocity_change']}%")
print(f"Velocity Assessment: {period_comparison['velocity_assessment']}")
print(f"\n💡 {period_comparison['interpretation']}")
print()

In [ ]:
# Long-Term Probability Assessment
print("\nLONG-TERM PROBABILITY ASSESSMENT:")
print("═══════════════════════════════════════════════════════════════")

# User's target is $0.83 breakeven
TARGET = 0.83
key_high_prices = [h[1] for h in key_highs]

probability = calculate_long_term_probability(
    current_price=0.74,
    target_price=TARGET,
    key_highs=key_high_prices,
    insider_avg_sell=insider['avg_sell_price']
)

print(f"Current Price: ${probability['current_price']}")
print(f"Target Price: ${probability['target_price']}")
print(f"Distance to Target: {probability['distance_to_target_pct']}%")
print(f"\nHISTORICAL CONTEXT:")
print(f"  Hit Rate: {probability['historical_hit_rate']}")
print(f"  Highs Above Target: {probability['highs_above_target']}/{probability['total_highs_analyzed']}")
print(f"  Insider Signal: {probability['insider_signal']}")
print(f"\nPROBABILITY: {probability['probability_assessment']}")
print("\n═══════════════════════════════════════════════════════════════")

## Interactive: Your Own 1-Year Analysis

Pas de waardes hieronder aan voor jouw eigen stock:

In [ ]:
# ==== PAS DEZE WAARDES AAN ====
MY_TICKER = "EXAMPLE"
MY_52W_HIGH = 50.00
MY_52W_LOW = 20.00
MY_CURRENT_PRICE = 35.00
MY_SHARES_START = 100_000_000
MY_SHARES_END = 120_000_000  # 20% dilution

# Maak je eigen analyzer
my_analyzer = OneYearAnalyzer(ticker=MY_TICKER)

my_analyzer.add_annual_metrics(
    week_52_high=MY_52W_HIGH,
    week_52_low=MY_52W_LOW,
    current_price=MY_CURRENT_PRICE,
    shares_outstanding_start=MY_SHARES_START,
    shares_outstanding_end=MY_SHARES_END,
    performance_ytd=5.0,
    performance_year=10.0,
    performance_half=-5.0
)

# Run analyses
print(f"1-YEAR ANALYSIS: {MY_TICKER}")
print("=" * 60)

dilution = my_analyzer.calculate_dilution_impact()
print(f"Dilution: {dilution['dilution_factor']}x — {dilution['severity']}")

position = my_analyzer.assess_52w_position()
print(f"Position: {position['position_quality']}")
print(f"Range: {position['range_position_pct']}th percentile")

# Simplified risk analysis
my_risk = my_analyzer.calculate_risk_asymmetry(
    upside_target=MY_52W_HIGH * 0.9,  # 90% of high
    downside_support=MY_52W_LOW * 1.2  # 20% above low
)
print(f"\nR/R to Support: {my_risk['rr_ratio_to_support']}:1")
print(f"Assessment: {my_risk['assessment']}")